In [11]:
import time
import numpy as np
import torch
import onnxruntime as ort
import pandas as pd

from preparation import get_model_paths, get_test_loader
from torchvision.models.detection import ssd300_vgg16
from torchvision.models import VGG16_Weights
import torch.nn.functional as F

In [12]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
paths = get_model_paths("model")

def build_model(num_classes: int):
    return ssd300_vgg16(
        weights=None,
        weights_backbone=VGG16_Weights.DEFAULT,
        num_classes=num_classes,
    )

# FP32
fp32_ckpt = torch.load(paths["basic_pth"], map_location=DEVICE)
fp32_model = build_model(fp32_ckpt["num_classes"]).to(DEVICE)
fp32_model.load_state_dict(fp32_ckpt["model_state_dict"])
fp32_model.eval()

# Quantized (저장된 quantized state_dict 로드)
q_ckpt = torch.load(paths["quant_pth"], map_location="cpu")
quant_model = torch.ao.quantization.quantize_dynamic(
    build_model(q_ckpt["num_classes"]).cpu().eval(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)
quant_model.load_state_dict(q_ckpt["model_state_dict"])
quant_model.eval()

# ONNX
ort_session = ort.InferenceSession(str(paths["onnx"]), providers=["CPUExecutionProvider"])
onnx_input_name = ort_session.get_inputs()[0].name

C:\Users\pixar\AppData\Local\Temp\ipykernel_5980\1672105911.py:19: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant_model = torch.ao.quantization.quantize_dynamic(


In [ ]:
print("onnx input:", ort_session.get_inputs()[0].shape)
print("onnx outputs:", [(o.name, o.shape, o.type) for o in ort_session.get_outputs()])

onnx input: [3, 300, 300]
onnx outputs: [('boxes', ['Concatboxes_dim_0', 4], 'tensor(float)'), ('labels', ['Gatherlabels_dim_0'], 'tensor(float)'), ('scores', ['Gatherlabels_dim_0'], 'tensor(int64)')]


In [14]:
test_loader = get_test_loader(batch_size=1)

In [19]:
def preprocess_to_300(img_t: torch.Tensor):
    # img_t: [3,H,W] float32 [0,1]
    x = img_t.unsqueeze(0)  # [1,3,H,W]
    x = F.interpolate(x, size=(300, 300), mode="bilinear", align_corners=False)
    return x.squeeze(0)     # [3,300,300]


def top1_label_from_det(out):
    # out: dict with boxes, labels, scores
    if len(out["scores"]) == 0:
        return 0
    i = int(torch.argmax(out["scores"]).item())
    return int(out["labels"][i].item())


def evaluate_torch(model, loader, device):
    correct, total = 0, 0
    start = time.perf_counter()

    with torch.no_grad():
        for images, labels, _ in loader:
            for img, y in zip(images, labels):
                out = model([img.to(device)])[0]
                pred = top1_label_from_det(out)
                if pred == int(y.item()):
                    correct += 1
                total += 1

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return correct / total, elapsed, (elapsed / total) * 1000


def evaluate_onnx(session, input_name, loader):
    correct, total = 0, 0
    start = time.perf_counter()

    for images, labels, _ in loader:
        for img, y in zip(images, labels):
            x = preprocess_to_300(img).cpu().numpy().astype(np.float32)  # [3,300,300]

            # 현재 모델은 outputs order가 (boxes, scores, labels)처럼 나옴
            boxes, scores_f, labels_i = session.run(None, {input_name: x})

            scores = np.asarray(scores_f).reshape(-1).astype(np.float32)
            pred_labels = np.asarray(labels_i).reshape(-1).astype(np.int64)

            if scores.size == 0:
                pred = 0
            else:
                pred = int(pred_labels[int(np.argmax(scores))])

            correct += int(pred == int(y.item()))
            total += 1

    elapsed = time.perf_counter() - start
    return correct / total, elapsed, (elapsed / total) * 1000


acc_onnx, t_onnx, ms_onnx = evaluate_onnx(ort_session, onnx_input_name, test_loader)
acc_fp32, t_fp32, ms_fp32 = evaluate_torch(fp32_model, test_loader, DEVICE)
acc_q, t_q, ms_q = evaluate_torch(quant_model, test_loader, torch.device("cpu"))

result = pd.DataFrame([
    {"model": "PyTorch FP32", "accuracy": acc_fp32, "total_sec": t_fp32, "ms_per_image": ms_fp32},
    {"model": "PyTorch Quantized", "accuracy": acc_q, "total_sec": t_q, "ms_per_image": ms_q},
    {"model": "ONNX", "accuracy": acc_onnx, "total_sec": t_onnx, "ms_per_image": ms_onnx},
])

result.sort_values("ms_per_image")

,model,accuracy,total_sec,ms_per_image
2,ONNX,0.528754,1102.497897,300.490024
0,PyTorch FP32,0.528754,1279.994535,348.867412
1,PyTorch Quantized,0.528754,1292.044794,352.151756
